In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!pip install -q pandas tqdm scikit-learn pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⬇️ EDIT THIS: put your ZIP file path inside Drive
zip_path = "/content/drive/MyDrive/Alzheimer_(Preprocessed Data).zip"

# No need to edit this unless you want a different folder name
extract_to = "/content/drive/MyDrive/dataset"

import zipfile, os
os.makedirs(extract_to, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)
print("Extracted to:", extract_to)


Mounted at /content/drive
Extracted to: /content/drive/MyDrive/dataset


In [ ]:
import os
root = "/content/drive/MyDrive/dataset"   # ⬅️ If you extracted elsewhere, change path

print("Root exists:", os.path.exists(root))
print("Top-level contents:", os.listdir(root))

# Check class folders and file counts
classes = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
print("Detected classes:", classes)
for c in classes:
    files = [f for f in os.listdir(os.path.join(root,c)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f" - {c}: {len(files)} images")


Root exists: True
Top-level contents: ['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented', 'Alzheimer (Preprocessed Data)']
Detected classes: ['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented', 'Alzheimer (Preprocessed Data)']
 - Mild_Demented: 896 images
 - Moderate_Demented: 64 images
 - Non_Demented: 3200 images
 - Very_Mild_Demented: 2240 images
 - Alzheimer (Preprocessed Data): 0 images


In [ ]:
import os, csv
from sklearn.model_selection import train_test_split

root = "/content/drive/MyDrive/dataset"    # ⬅️ Path where your class folders are - Corrected path

# 🔒 FIXED order (manually define, do not sort!)
classes = ["Non_Demented", "Very_Mild_Demented", "Mild_Demented", "Moderate_Demented"]

# Assign custom labels
class_map = {
    "Non_Demented": 0,
    "Very_Mild_Demented": 1,
    "Mild_Demented": 2,
    "Moderate_Demented": 3
}
print("Class mapping (label assigned):", class_map)

rows = []
for c in classes:
    class_folder = os.path.join(root, c)
    for fname in os.listdir(class_folder):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append([os.path.join(class_folder, fname), class_map[c]])

if len(rows) == 0:
    raise SystemExit("⚠️ No jpg/png found. Check folder path or file types.")

train_rows, val_rows = train_test_split(
    rows, test_size=0.2, random_state=42,
    stratify=[r[1] for r in rows]
)

out_dir = "/content/data"  # ⬅️ Change only if you want another folder
os.makedirs(out_dir, exist_ok=True)

def save_csv(path, rows):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["mri_path", "label"])
        w.writerows(rows)

save_csv(os.path.join(out_dir, "train.csv"), train_rows)
save_csv(os.path.join(out_dir, "val.csv"), val_rows)

print(f"✅ train.csv = {len(train_rows)}, val.csv = {len(val_rows)}")

Class mapping (label assigned): {'Non_Demented': 0, 'Very_Mild_Demented': 1, 'Mild_Demented': 2, 'Moderate_Demented': 3}
✅ train.csv = 5120, val.csv = 1280


In [ ]:
import os, random, pandas as pd, numpy as np
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.optim import Adam
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from tqdm import tqdm

# ---------------- Config ----------------
data_dir = "/content/data"  # ⬅️ Path to train.csv & val.csv
train_csv = os.path.join(data_dir, "train.csv")
val_csv = os.path.join(data_dir, "val.csv")
img_size = 224
batch_size = 16   # ⬅️ Lower if GPU runs out of memory
epochs = 15
lr = 1e-4
checkpoint_path = "/content/best_resnet18.pt"
num_workers = 2


In [ ]:
# Set seed
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Get number of classes from CSV
df = pd.read_csv(train_csv)
num_classes = int(df['label'].nunique())
print("Num classes:", num_classes)

# Dataset class
class MRIJPGDataset(Dataset):
    def __init__(self, csv_file, img_size=224, augment=False):
        self.data = pd.read_csv(csv_file)
        self.transform = transforms.Compose([
            transforms.Resize((img_size,img_size)),
            transforms.RandomHorizontalFlip() if augment else transforms.Lambda(lambda x: x),
            transforms.RandomRotation(8) if augment else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize([0.485],[0.229])
        ])
        self.augment = augment
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row['mri_path']).convert("L")
        img = self.transform(img)
        img = img.repeat(3,1,1)   # grayscale → 3 channels
        return img, int(row['label'])

train_ds = MRIJPGDataset(train_csv, img_size=img_size, augment=True)
val_ds   = MRIJPGDataset(val_csv, img_size=img_size, augment=False)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# Model
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=lr)


Device: cpu
Num classes: 4


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 145MB/s]


In [ ]:
best_auc = 0.0
for epoch in range(1, epochs+1):
    # train
    model.train()
    train_losses, preds, trues = [], [], []
    for x,y in tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False):
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
        trues.extend(y.cpu().tolist())
    train_loss = np.mean(train_losses)

    # val
    model.eval()
    val_losses, preds_v, trues_v = [], [], []
    with torch.no_grad():
        for x,y in tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False):
            x,y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_losses.append(loss.item())
            preds_v.extend(torch.argmax(logits, dim=1).cpu().tolist())
            trues_v.extend(y.cpu().tolist())
    val_loss = np.mean(val_losses)
    acc = accuracy_score(trues_v, preds_v)
    f1 = f1_score(trues_v, preds_v, average='weighted', zero_division=0)

    print(f"Epoch {epoch}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f} acc={acc:.4f} f1={f1:.4f}")

    if acc > best_auc:
        best_auc = acc
        torch.save(model.state_dict(), checkpoint_path)
        print("✅ Saved best model:", checkpoint_path)

print("Training finished. Best val acc:", best_auc)


Epoch 1 train:   7%|▋         | 23/320 [02:19<19:59,  4.04s/it]

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

def plot_confusion_matrix(model, dataloader, device, class_names):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    print("Classification Report:\n", classification_report(all_labels, all_preds, target_names=class_names))

    # Plot
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()


In [ ]:
class_names = ["Non_Demented", "Very_Mild_Demented", "Mild_Demented", "Moderate_Demented"]
plot_confusion_matrix(model, val_loader, device, class_names)


In [ ]:
from torchvision import transforms
from PIL import Image
import torch

# Same transforms as training/validation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ResNet pretrained mean
                         [0.229, 0.224, 0.225])  # ResNet pretrained std
])


In [ ]:
import torch.nn.functional as F

def predict_image(model, image_path, device, class_names):
    # Load and preprocess image
    img = Image.open(image_path).convert("RGB")
    img_t = transform(img).unsqueeze(0).to(device)  # add batch dimension

    # Model inference
    model.eval()
    with torch.no_grad():
        outputs = model(img_t)
        probs = F.softmax(outputs, dim=1)
        conf, pred_class = torch.max(probs, 1)

    predicted_label = class_names[pred_class.item()]
    confidence = conf.item()

    return predicted_label, confidence


In [ ]:
import matplotlib.pyplot as plt

def predict_and_show(model, image_path, device, class_names):
    # Load and preprocess
    img = Image.open(image_path).convert("RGB")
    img_t = transform(img).unsqueeze(0).to(device)

    # Model inference
    model.eval()
    with torch.no_grad():
        outputs = model(img_t)
        probs = F.softmax(outputs, dim=1)
        conf, pred_class = torch.max(probs, 1)

    predicted_label = class_names[pred_class.item()]
    confidence = conf.item()

    # Show image + prediction
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predicted: {predicted_label}\nConfidence: {confidence:.2f}")
    plt.show()

    return predicted_label, confidence


In [ ]:
class_names = ["Non_Demented", "Very_Mild_Demented", "Mild_Demented", "Moderate_Demented"]

image_path = "/content/drive/MyDrive/Test4.png"   # 👈 Replace with your MRI file path
label, conf = predict_and_show(model, image_path, device, class_names)

print(f"Prediction -> {label} (Confidence: {conf:.2f})")


In [ ]:
torch.save(model, "resnet18_dementia_full.pth")
print("Full model saved as resnet18_dementia_full.pth")
